In [133]:
import pandas as pd
import calendar
import random
import numpy as np
from calendar import monthrange

USE_RANDOM_SEED = True
RANDOM_SEED = 42

def reset_tiebreak_rng():
    return random.Random(RANDOM_SEED) if USE_RANDOM_SEED else random.Random()

tiebreak_rng = reset_tiebreak_rng()

Important Notes:
- Matching of clerk details among different csv is done by searching up the clerk name without the rank (rank can change anytime)

In [134]:
availability = pd.read_csv("april_availability.csv")
points = pd.read_csv("march_points.csv")

In [135]:
points.head()

,Name,Duty
0,SHYEIKH SALEH HUSSAIN BIN MOHAMMED SULAIMAN,0.0
1,MUHAMMAD HILAL IZDIHAR BIN MOHAMAD IZZUL SHAH,2.0
2,OH EE BING,2.0
3,JEREMIAH FOO JI SHUAN,1.0
4,CHEONG KAI JIE,1.0


In [136]:
availability.head()

,Name,01/04/26,02/04/26,03/04/26,04/04/26,05/04/26,06/04/26,07/04/26,08/04/26,09/04/26,...,22/04/26,23/04/26,24/04/26,25/04/26,26/04/26,27/04/26,28/04/26,29/04/26,30/04/26,Availability
0,SIDDARTH SREEKUMAR CHERUBAL,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,30
1,CLARENCE ONG JUN YI,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,XINHANG,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,SONG HAOYAN,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,DARYL SEOW SHI YU,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,30


## Computing Duty Clerk Point for the month

1. <b>Extended Leave.</b> If a clerk skips a month e.g. re-BMT, how will his points be calculated? Would it be the 3 most recent months?

Ans: Do not keep life-time points. If clerk skips a month, duty points will be reset to N.A.

2. <b>Procedure to assign duty. </b>
    - Assign all available duty clerks the minimum 1 duty point for the month
    - Assign the remaining duty points, based on the total duties done in the past 3 months, recent activation and "credibility ratings". Limit duty points assigned to 2
    - Let Ryan adjust the points
    - Input projected points, availabilities and preferences to algorithm

3. <b>New Clerk.</b> When a new clerk joins the roster, is there a way to decide whether he does 1 or 2 duty in the month? Should a 2nd duty be assigned to current clerks who have yet to meet their obligation first or to the new clerk first?

Ans: New Clerk does 1 duty points in the second half of the month, ideally 1 week break from the understudy period. Works out to be 1st or 2nd week: Understudy duty, followed by 3th/4th week: Actual Duty


In [137]:
# Calculate Required Duty Points for the month
YEAR = 2026
MONTH = 4
_, last_day = monthrange(YEAR, MONTH)

days = pd.date_range(start=f'2026-{MONTH:2d}-01', periods=last_day)
weekdays = np.sum(days.weekday < 5)
weekends = np.sum(days.weekday >= 5)

DUTY = weekdays * 1 + weekends * 2
print(DUTY)

38


In [138]:
planning_table = availability.copy()

# A clerk is assigned duty in a month only if they are free on at least 1 days
excluded_clerks = planning_table[planning_table["Availability"]==0]
planning_table = planning_table[planning_table["Availability"]>0]

In [139]:
excluded_clerks.head()

,Name,01/04/26,02/04/26,03/04/26,04/04/26,05/04/26,06/04/26,07/04/26,08/04/26,09/04/26,...,22/04/26,23/04/26,24/04/26,25/04/26,26/04/26,27/04/26,28/04/26,29/04/26,30/04/26,Availability
1,CLARENCE ONG JUN YI,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,XINHANG,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,SONG HAOYAN,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [140]:
clerks = planning_table["Name"].tolist()

In [141]:
# Find duty points record for all clerk
for clerk in clerks:
    if clerk not in points["Name"].tolist():
        print(f"> No matching points found for {clerk}")
        
        # Assume the clerks are new
        # new_entry = pd.DataFrame([[clerk, 0.00]], columns=["Name", "Duty"]) # fill values for clerk not found
        # pd.concat([points, new_entry])

        # Assume the clerks ord
        planning_table = planning_table[planning_table["Name"]!=clerk]
        
points.head()

> No matching points found for JULLION THNG AIK HONG
> No matching points found for AKMAL AHMAD PUTRA BIN SAMAD
> No matching points found for MOHAMED ROSHAN AKTAR BIN AZEES ALI
> No matching points found for RAFALE


,Name,Duty
0,SHYEIKH SALEH HUSSAIN BIN MOHAMMED SULAIMAN,0.0
1,MUHAMMAD HILAL IZDIHAR BIN MOHAMAD IZZUL SHAH,2.0
2,OH EE BING,2.0
3,JEREMIAH FOO JI SHUAN,1.0
4,CHEONG KAI JIE,1.0


In [142]:
# Add duty points to planning table
planning_table = planning_table.merge(points[["Name", "Duty"]], on="Name", how="left", sort=False)

In [143]:
# Simple Projection
curr_duty = planning_table["Duty"].to_numpy()

# clerk list
clerk = planning_table["Name"]

# Assign 1 duty point per clerk
projected_duty = [1.0 if duty < 4.0 else 0.00 for duty in curr_duty]

planning_table["Projected"] = projected_duty

# Display
for name, duty in zip(clerk, projected_duty):
    print(f"{name}: {duty}")

SIDDARTH SREEKUMAR CHERUBAL: 1.0
DARYL SEOW SHI YU: 1.0
MUHAMMAD NORSYAFIQ BIN SELAMAT: 1.0
OH EE BING: 1.0
MUHAMMAD NUR HAKIM BIN SHAHARIZIN: 1.0
SURIAPRAKASH THRIUVALAKU: 1.0
FATHUL HAKIM BIN MUHAMMAD FAUZI: 1.0
EBRAHIM HAKEEM BIN AMEEHAR: 1.0
ANISH VINAAYAK SIVASANKAR: 1.0
DYLAN TAN: 1.0
SHAVIN VALEN: 1.0
CHEONG JUN KAI, HARRY: 1.0
JEREMIAH FOO JI SHUAN: 1.0
CHEONG KAI JIE: 1.0
GUO JUNQI LEO: 1.0
CHONG SIANG NATHAN: 1.0
BRYAN MARC MEHAR: 1.0
SHYEIKH SALEH HUSSAIN BIN MOHAMMED SULAIMAN: 1.0
MUHAMMAD HILAL IZDIHAR BIN MOHAMAD IZZUL SHAH: 1.0
MITCHELL LIM: 1.0
HUANG JIANHAO: 0.0
RYAN TAY RUI YANG: 1.0
ISAAC LOH KA LOK: 1.0
NICO CHUA WEE FONG: 1.0
JERRYL TAN MING JIE: 1.0
MUHAMAD RIFQY AQIL BIN MUHAMAD SHAH: 1.0
AUSTIN MAXIMILLIAN KOLBE LIM KIM: 1.0


In [144]:
planning_table.head()

,Name,01/04/26,02/04/26,03/04/26,04/04/26,05/04/26,06/04/26,07/04/26,08/04/26,09/04/26,...,24/04/26,25/04/26,26/04/26,27/04/26,28/04/26,29/04/26,30/04/26,Availability,Duty,Projected
0,SIDDARTH SREEKUMAR CHERUBAL,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,30,0.0,1.0
1,DARYL SEOW SHI YU,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,30,1.0,1.0
2,MUHAMMAD NORSYAFIQ BIN SELAMAT,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,30,0.0,1.0
3,OH EE BING,1,1,1,1,0,0,0,0,0,...,1,1,1,1,1,1,1,22,2.0,1.0
4,MUHAMMAD NUR HAKIM BIN SHAHARIZIN,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,30,0.0,1.0


In [145]:
# Assign remaining duty points fairly
import heapq

assigned = int(planning_table["Projected"].sum())
print(f"Current Assigned Duty points is {assigned}")
tiebreak_rng = reset_tiebreak_rng()
heap =[(-point, tiebreak_rng.random(), clerk) for point, clerk in zip((4-(1+curr_duty)).tolist(), clerk)] # calculate remaining duty

heapq.heapify(heap)

for i in range(DUTY-assigned):
    neg_count, _, name = heapq.heappop(heap)
    planning_table.loc[planning_table["Name"]==name, "Projected"] += 1.00

for name, projected in zip(planning_table["Name"], planning_table["Projected"]):
    print(f"{name}: {projected}")

final = int(planning_table["Projected"].sum())
print(f"Adjusted Duty points is {final}")


Current Assigned Duty points is 26
SIDDARTH SREEKUMAR CHERUBAL: 2.0
DARYL SEOW SHI YU: 1.0
MUHAMMAD NORSYAFIQ BIN SELAMAT: 2.0
OH EE BING: 1.0
MUHAMMAD NUR HAKIM BIN SHAHARIZIN: 2.0
SURIAPRAKASH THRIUVALAKU: 2.0
FATHUL HAKIM BIN MUHAMMAD FAUZI: 2.0
EBRAHIM HAKEEM BIN AMEEHAR: 1.0
ANISH VINAAYAK SIVASANKAR: 1.0
DYLAN TAN: 1.0
SHAVIN VALEN: 1.0
CHEONG JUN KAI, HARRY: 2.0
JEREMIAH FOO JI SHUAN: 1.0
CHEONG KAI JIE: 1.0
GUO JUNQI LEO: 1.0
CHONG SIANG NATHAN: 2.0
BRYAN MARC MEHAR: 2.0
SHYEIKH SALEH HUSSAIN BIN MOHAMMED SULAIMAN: 2.0
MUHAMMAD HILAL IZDIHAR BIN MOHAMAD IZZUL SHAH: 1.0
MITCHELL LIM: 2.0
HUANG JIANHAO: 0.0
RYAN TAY RUI YANG: 1.0
ISAAC LOH KA LOK: 1.0
NICO CHUA WEE FONG: 2.0
JERRYL TAN MING JIE: 1.0
MUHAMAD RIFQY AQIL BIN MUHAMAD SHAH: 1.0
AUSTIN MAXIMILLIAN KOLBE LIM KIM: 2.0
Adjusted Duty points is 38


In [146]:
planning_table.head()

,Name,01/04/26,02/04/26,03/04/26,04/04/26,05/04/26,06/04/26,07/04/26,08/04/26,09/04/26,...,24/04/26,25/04/26,26/04/26,27/04/26,28/04/26,29/04/26,30/04/26,Availability,Duty,Projected
0,SIDDARTH SREEKUMAR CHERUBAL,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,30,0.0,2.0
1,DARYL SEOW SHI YU,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,30,1.0,1.0
2,MUHAMMAD NORSYAFIQ BIN SELAMAT,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,30,0.0,2.0
3,OH EE BING,1,1,1,1,0,0,0,0,0,...,1,1,1,1,1,1,1,22,2.0,1.0
4,MUHAMMAD NUR HAKIM BIN SHAHARIZIN,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,30,0.0,2.0


In [147]:
planning_table.to_csv("planning_table.csv", index=False)

## Plan for the month

In [148]:
YEAR = 2026
MONTH = 4
DAYS = calendar.monthrange(YEAR, MONTH)[1]

In [149]:
# compute weight
# weight_1 * preference + weight_2 * duty_count + weight_3 * weekend

# Algorithm should space out 2 duties as much as possible

In [150]:
# Update Planning Table
# 1. Duty Point this month
# 2. Planning Weight
m, n = planning_table.shape
planning_table["Monthly Duty Points"] = [0 for _ in range(m)]
planning_table["Planning Weight"] = [None for _ in range(m)]

In [151]:
planning_sequence = [f"{day:02d}/{MONTH:02d}/26" for day in range(1, DAYS+1)]
print(planning_sequence)

['01/04/26', '02/04/26', '03/04/26', '04/04/26', '05/04/26', '06/04/26', '07/04/26', '08/04/26', '09/04/26', '10/04/26', '11/04/26', '12/04/26', '13/04/26', '14/04/26', '15/04/26', '16/04/26', '17/04/26', '18/04/26', '19/04/26', '20/04/26', '21/04/26', '22/04/26', '23/04/26', '24/04/26', '25/04/26', '26/04/26', '27/04/26', '28/04/26', '29/04/26', '30/04/26']


In [152]:
planning_table.to_csv("planning_table.csv", index=False)

In [153]:
planning_table

,Name,01/04/26,02/04/26,03/04/26,04/04/26,05/04/26,06/04/26,07/04/26,08/04/26,09/04/26,...,26/04/26,27/04/26,28/04/26,29/04/26,30/04/26,Availability,Duty,Projected,Monthly Duty Points,Planning Weight
0,SIDDARTH SREEKUMAR CHERUBAL,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,30,0.0,2.0,0,None
1,DARYL SEOW SHI YU,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,30,1.0,1.0,0,None
2,MUHAMMAD NORSYAFIQ BIN SELAMAT,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,30,0.0,2.0,0,None
3,OH EE BING,1,1,1,1,0,0,0,0,0,...,1,1,1,1,1,22,2.0,1.0,0,None
4,MUHAMMAD NUR HAKIM BIN SHAHARIZIN,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,30,0.0,2.0,0,None
5,SURIAPRAKASH THRIUVALAKU,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,30,1.0,2.0,0,None
6,FATHUL HAKIM BIN MUHAMMAD FAUZI,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,30,1.0,2.0,0,None
7,EBRAHIM HAKEEM BIN AMEEHAR,1,1,1,0,0,1,1,1,1,...,0,1,1,1,1,22,2.0,1.0,0,None
8,ANISH VINAAYAK SIVASANKAR,1,1,1,1,1,0,0,0,0,...,1,1,1,1,1,24,3.0,1.0,0,None
9,DYLAN TAN,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,29,2.0,1.0,0,None


In [ ]:
def compute_weight(date):
    weights = []
    for row in range(m):
        duty_point = 0

        availability = planning_table.loc[row, date]
        if not availability: duty_point = -1
        else: duty_point += 1

        projected = planning_table.loc[row, "Projected"]
        month = planning_table.loc[row, "Monthly Duty Points"]
        duty_count = projected - month
        if duty_count == 0: duty_point = -1
        else: duty_point += duty_count

        name = planning_table.loc[row, "Name"]
        # Keep higher duty weights first, but randomize ties instead of falling back to alphabetical order.
        heapq.heappush(weights, (-duty_point, tiebreak_rng.random(), name))
    
    return weights

def dfs(i):
    if i == len(planning_sequence):
        return True

    weights = compute_weight(planning_sequence[i])

    while weights:
        duty_points, _, name = heapq.heappop(weights)
        if -duty_points > 0:
            print(f"On {planning_sequence[i]}, {name}, {duty_points} is selected")
            row = planning_table.index[planning_table["Name"] == name].item()
            planning_table.loc[row, "Monthly Duty Points"] += 1
            if dfs(i+1):
                return True
            print("Planning Failed. Backtracking")
            planning_table.loc[row, "Monthy Duty Points"] -= 1
    
    return False

tiebreak_rng = reset_tiebreak_rng()
dfs(0)

On 01/04/26, CHEONG JUN KAI, HARRY is selected
On 02/04/26, MUHAMMAD NORSYAFIQ BIN SELAMAT is selected
On 03/04/26, MUHAMMAD NUR HAKIM BIN SHAHARIZIN is selected
On 04/04/26, AUSTIN MAXIMILLIAN KOLBE LIM KIM is selected
On 05/04/26, SHYEIKH SALEH HUSSAIN BIN MOHAMMED SULAIMAN is selected
On 06/04/26, SIDDARTH SREEKUMAR CHERUBAL is selected
On 07/04/26, MITCHELL LIM is selected
On 08/04/26, FATHUL HAKIM BIN MUHAMMAD FAUZI is selected
On 09/04/26, SURIAPRAKASH THRIUVALAKU is selected
On 10/04/26, BRYAN MARC MEHAR is selected
On 11/04/26, NICO CHUA WEE FONG is selected
On 12/04/26, CHONG SIANG NATHAN is selected
On 13/04/26, SURIAPRAKASH THRIUVALAKU is selected
On 14/04/26, MUHAMMAD HILAL IZDIHAR BIN MOHAMAD IZZUL SHAH is selected
On 15/04/26, MUHAMMAD NUR HAKIM BIN SHAHARIZIN is selected
On 16/04/26, SIDDARTH SREEKUMAR CHERUBAL is selected
On 17/04/26, RYAN TAY RUI YANG is selected
On 18/04/26, ISAAC LOH KA LOK is selected
On 19/04/26, CHONG SIANG NATHAN is selected
On 20/04/26, SHYEIKH 

True